In [0]:

# POC-2: Validate CDC sequence compatibility
# READ ONLY — no tables or files are modified.

from pyspark.sql import functions as F

BASE = "/Volumes/retail_poc2/landing/source_files"

initial = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/initial/orders.csv")
    .select("order_id", "source_sequence")
)

cdc = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/CDC/orders_cdc_batch_*.csv")
    .select(
        "event_id",
        "order_id",
        "operation",
        "event_sequence",
        "event_timestamp"
    )
)

print("1. INITIAL ORDER SEQUENCE RANGE")
initial.select(
    F.min("source_sequence").alias("min_sequence"),
    F.max("source_sequence").alias("max_sequence")
).show()

print("2. CDC SEQUENCE RANGE")
cdc.select(
    F.min("event_sequence").alias("min_sequence"),
    F.max("event_sequence").alias("max_sequence")
).show()

print("3. CDC EVENTS MATCHING INITIAL ORDERS")
matched = cdc.join(initial, "order_id", "inner")

matched.select(
    "order_id",
    "source_sequence",
    "event_sequence",
    "event_timestamp",
    "operation"
).orderBy("order_id", "event_sequence").show(
    50, truncate=False
)

print("4. CDC EVENTS NOT NEWER THAN INITIAL STATE")
matched.filter(
    F.col("event_sequence") <= F.col("source_sequence")
).show(truncate=False)

print("5. DUPLICATE CDC EVENT IDS")
cdc.groupBy("event_id").count().filter(
    F.col("count") > 1
).show(truncate=False)

print("6. EQUAL-SEQUENCE AND TIMESTAMP CONFLICTS")
cdc.groupBy(
    "order_id", "event_sequence", "event_timestamp"
).agg(
    F.count("*").alias("event_count"),
    F.countDistinct("event_id").alias("distinct_event_ids")
).filter(
    F.col("event_count") > 1
).show(truncate=False)

print("7. CDC EVENTS WITH NULL ORDERING FIELDS")
cdc.filter(
    F.col("order_id").isNull()
    | F.col("event_id").isNull()
    | F.col("event_sequence").isNull()
    | F.col("event_timestamp").isNull()
).show(truncate=False)

print("READ-ONLY CDC ORDERING INSPECTION COMPLETED")
